In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from dataclasses import dataclass
import copy
import PIL

In [2]:
@dataclass
class Config:
    batch_size = 256
    num_workers = 8
    pin_memory = True
    persistent_workers = True
    prefetch_factors = 4
    lr = 1e-3
    weight_decay = 1e-4
    num_epochs = 100
    val_split = 500
    train_split = 50000 - val_split
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3), value="random"),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD), 
    ]
)


    
cfg = Config()

dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
train_set, val_set = torch.utils.data.random_split(dataset, [cfg.train_split, cfg.val_split])

train_loader = torch.utils.data.DataLoader(
    train_set, 
    batch_size=cfg.batch_size, 
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers,
    prefetch_factor=cfg.prefetch_factors,
)


val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=cfg.batch_size
)

test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
test_loader = torch.utils.data.DataLoader(
    test_set, 
    batch_size=256,
    shuffle=False,
    num_workers=cfg.num_workers,
)


Files already downloaded and verified
Files already downloaded and verified


In [3]:
class SignSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.where(x >= 0, torch.ones_like(x), -torch.ones_like(x))

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * (x.abs() <= 1).to(grad_output.dtype)


class BinaryActivation(nn.Module):
    def forward(self, x):
        return SignSTE.apply(x)

In [4]:
class XNORLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.kaiming_normal_(self.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x):
        bx = SignSTE.apply(x)
        bw = SignSTE.apply(self.weight)

        alpha = self.weight.abs().mean(dim=1, keepdim=True).view(1, -1)
        beta = x.detach().abs().mean(dim=1, keepdim=True)

        return F.linear(bx, bw, self.bias) * alpha * beta

In [5]:
class XNORConv2d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        padding=0,
        dilation=1,
        groups=1,
        bias=False,
    ):
        super().__init__()

        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        self.groups = groups

        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)

        self.kernel_size = kernel_size

        self.weight = nn.Parameter(
            torch.empty(out_channels, in_channels // groups, *kernel_size)
        )

        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None

        nn.init.kaiming_normal_(self.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x):
        bx = SignSTE.apply(x)
        bw = SignSTE.apply(self.weight)

        alpha = self.weight.abs().mean(dim=(1, 2, 3), keepdim=True)

        beta = F.avg_pool2d(
            x.detach().abs(),
            kernel_size=self.kernel_size,
            stride=self.stride,
            padding=self.padding,
        )

        beta = beta.mean(dim=1, keepdim=True)

        out = F.conv2d(
            bx,
            bw,
            self.bias,
            stride=self.stride,
            padding=self.padding,
            dilation=self.dilation,
            groups=self.groups,
        )

        return out * alpha.view(1, -1, 1, 1) * beta

In [6]:
class Network(nn.Module):
    def __init__(self, input_channels=3, num_classes=10, image_size=32, dropout=0.25, n_ratio=1.0):
        super().__init__()
        self.img_size = image_size
        # self.fcin = self.img_size**2
        self.cin = input_channels
        self.cout1 = 32
        self.cout2 = 32

        self.fcin = self.cout2 * (16**2)
        self.fcout1 = 128
        self.fcout2 = 128
        self.fcout3 = num_classes

        self.n_ratio=n_ratio

        self.block1 = nn.Sequential(
            nn.Conv2d(self.cin, self.cout1, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(self.cout1),
            nn.ReLU(),
            nn.MaxPool2d((2,2))
            # nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.BatchNorm2d(self.cout1),
            BinaryActivation(),
            XNORConv2d(self.cout1, self.cout2, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(self.cout2),
            BinaryActivation(),
        )
        self.block3 = nn.Sequential(
            nn.BatchNorm1d(self.fcin),
            BinaryActivation(),
            XNORLinear(self.fcin, self.fcout1),
            nn.BatchNorm1d(self.fcout1),
            BinaryActivation(),
            nn.Dropout(dropout),
        )
        self.block4 = nn.Sequential(
            nn.BatchNorm1d(self.fcout1),
            BinaryActivation(),
            XNORLinear(self.fcout1, self.fcout2,),
            nn.BatchNorm1d(self.fcout2),
            BinaryActivation(),
            nn.Dropout(dropout),
        )

        self.head = nn.Linear(self.fcout2, self.fcout3)

    def forward(self, x):

        
        x = self.block1(x)
        x = self.block2(x)
    
        x = torch.flatten(x, start_dim=1)
    
        x = self.block3(x)
        x = self.block4(x)
        x = self.head(x)
    
        return x

    def xnor_layers(self):
        return [m for m in self.modules() if isinstance(m, (XNORLinear, XNORConv2d))]

    

In [7]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch, labels in loader:
            batch, labels = batch.to(device), labels.to(device)
    
            outputs = model(batch)
            loss = criterion(outputs, labels)
    
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
    
            _, predicted = torch.max(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += batch_size

    avg_loss = total_loss / total
    avg_acc = correct / total

    return avg_loss, avg_acc

In [8]:
cfg = Config()
model = Network(n_ratio=5.0).to(cfg.device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

best_val_acc = 0.0
best_state_dict = None

for epoch in range(cfg.num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch, labels in train_loader:
        batch, labels = batch.to(cfg.device), labels.to(cfg.device)

        optimizer.zero_grad()

        outputs = model(batch)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    train_loss = running_loss / total
    train_acc = correct / total

    val_loss, val_acc = evaluate(model, val_loader, criterion, cfg.device,)

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

    print(
        f'Epoch [{epoch + 1}/{cfg.num_epochs}] '
        f'Train loss: {train_loss:.4f} '
        f'Train acc: {train_acc:.4f} '
        f'Val loss: {val_loss: .4f} '
        f'Val acc: {val_acc: .4f} '
    )

print(f'Best Validation Accuracy: {best_val_acc:.4f}')

model.load_state_dict(best_state_dict)
clean_state_dict = copy.deepcopy(model.state_dict())

clean_test_loss, clean_test_acc = evaluate(model, test_loader, criterion, cfg.device,)
print(
    f'Clean Test Loss: {clean_test_loss:.4f} '
    f'Clean Test Acc: {clean_test_acc:.4f}'
)


Epoch [1/100] Train loss: 2.0859 Train acc: 0.2256 Val loss:  1.7173 Val acc:  0.3640 
Epoch [2/100] Train loss: 1.7257 Train acc: 0.3592 Val loss:  1.6691 Val acc:  0.3400 
Epoch [3/100] Train loss: 1.6322 Train acc: 0.4028 Val loss:  1.6711 Val acc:  0.4200 
Epoch [4/100] Train loss: 1.5748 Train acc: 0.4273 Val loss:  1.5413 Val acc:  0.4660 
Epoch [5/100] Train loss: 1.5236 Train acc: 0.4498 Val loss:  1.4282 Val acc:  0.4960 
Epoch [6/100] Train loss: 1.5045 Train acc: 0.4564 Val loss:  1.5131 Val acc:  0.4520 
Epoch [7/100] Train loss: 1.4970 Train acc: 0.4615 Val loss:  1.4315 Val acc:  0.4920 
Epoch [8/100] Train loss: 1.4797 Train acc: 0.4648 Val loss:  1.4976 Val acc:  0.4780 
Epoch [9/100] Train loss: 1.4738 Train acc: 0.4706 Val loss:  1.4313 Val acc:  0.5200 
Epoch [10/100] Train loss: 1.4648 Train acc: 0.4764 Val loss:  1.5000 Val acc:  0.4540 
Epoch [11/100] Train loss: 1.4585 Train acc: 0.4784 Val loss:  1.5240 Val acc:  0.4860 
Epoch [12/100] Train loss: 1.4662 Train a

In [9]:
@torch.no_grad()
def flip_xnor_weights_global(model, pct: float, seed: int = 0):
    """
    Flip pct% of all real-valued underlying weights in XNORLinear layers.

    Since XNORLinear uses SignSTE(weight) in forward(),
    multiplying selected real weights by -1 flips their binarized sign.
    """
    assert 0 <= pct <= 100

    xnor_layers = model.xnor_layers()

    # Count total binarized weights across all XNORLinear layers
    total_weights = sum(layer.weight.numel() for layer in xnor_layers)
    num_to_flip = int(round(total_weights * pct / 100.0))

    if num_to_flip == 0:
        return 0

    # Create one global random list of indices across all XNOR layers
    device = xnor_layers[0].weight.device
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    flip_indices = torch.randperm(total_weights, device=device, generator=gen)[:num_to_flip]

    # Flip selected weights layer by layer
    offset = 0
    flipped = 0

    for layer in xnor_layers:
        w = layer.weight
        flat = w.view(-1)
        n = flat.numel()

        mask = (flip_indices >= offset) & (flip_indices < offset + n)
        local_indices = flip_indices[mask] - offset

        if local_indices.numel() > 0:
            flat[local_indices] *= -1.0
            flipped += local_indices.numel()

        offset += n

    return flipped

In [10]:
perturbation_results = []

for pct in range(0, 50 + 1, 5):
    # Always start from the clean trained model
    model.load_state_dict(clean_state_dict)
    model.to(cfg.device)
    model.eval()

    flipped = flip_xnor_weights_global(
        model,
        pct=pct,
        seed=10 + pct,
    )

    test_loss, test_acc = evaluate(model, test_loader, criterion, cfg.device)

    perturbation_results.append({
        "flipped_pct": pct,
        "num_flipped": flipped,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })

    print(
        f"Flipped {pct:02d}% | "
        f"Weights flipped: {flipped} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.4f}"
    )

Flipped 00% | Weights flipped: 0 | Test Loss: 1.0322 | Test Acc: 0.6422
Flipped 05% | Weights flipped: 53709 | Test Loss: 1.2520 | Test Acc: 0.5691
Flipped 10% | Weights flipped: 107418 | Test Loss: 1.6832 | Test Acc: 0.4103
Flipped 15% | Weights flipped: 161126 | Test Loss: 1.8432 | Test Acc: 0.3632
Flipped 20% | Weights flipped: 214835 | Test Loss: 2.2626 | Test Acc: 0.2323
Flipped 25% | Weights flipped: 268544 | Test Loss: 2.2516 | Test Acc: 0.2231
Flipped 30% | Weights flipped: 322253 | Test Loss: 2.2740 | Test Acc: 0.2287
Flipped 35% | Weights flipped: 375962 | Test Loss: 2.4377 | Test Acc: 0.1351
Flipped 40% | Weights flipped: 429670 | Test Loss: 2.5847 | Test Acc: 0.1050
Flipped 45% | Weights flipped: 483379 | Test Loss: 2.5563 | Test Acc: 0.1017
Flipped 50% | Weights flipped: 537088 | Test Loss: 2.4644 | Test Acc: 0.0992


In [11]:
cfg = Config()
model = Network(n_ratio=1.0).to(cfg.device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

best_val_acc = 0.0
best_state_dict = None

for epoch in range(cfg.num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch, labels in train_loader:
        batch, labels = batch.to(cfg.device), labels.to(cfg.device)

        optimizer.zero_grad()

        outputs = model(batch)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    train_loss = running_loss / total
    train_acc = correct / total

    val_loss, val_acc = evaluate(model, val_loader, criterion, cfg.device,)

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

    print(
        f'Epoch [{epoch + 1}/{cfg.num_epochs}] '
        f'Train loss: {train_loss:.4f} '
        f'Train acc: {train_acc:.4f} '
        f'Val loss: {val_loss: .4f} '
        f'Val acc: {val_acc: .4f} '
    )

print(f'Best Validation Accuracy: {best_val_acc:.4f}')

model.load_state_dict(best_state_dict)
clean_state_dict = copy.deepcopy(model.state_dict())

clean_test_loss, clean_test_acc = evaluate(model, test_loader, criterion, cfg.device,)
print(
    f'Clean Test Loss: {clean_test_loss:.4f} '
    f'Clean Test Acc: {clean_test_acc:.4f}'
)


Epoch [1/100] Train loss: 2.0898 Train acc: 0.2219 Val loss:  1.8407 Val acc:  0.3080 
Epoch [2/100] Train loss: 1.7315 Train acc: 0.3558 Val loss:  1.6545 Val acc:  0.4120 
Epoch [3/100] Train loss: 1.6358 Train acc: 0.3986 Val loss:  1.5785 Val acc:  0.4460 
Epoch [4/100] Train loss: 1.5861 Train acc: 0.4228 Val loss:  1.5154 Val acc:  0.4440 
Epoch [5/100] Train loss: 1.5576 Train acc: 0.4339 Val loss:  1.5218 Val acc:  0.4640 
Epoch [6/100] Train loss: 1.5422 Train acc: 0.4410 Val loss:  1.5590 Val acc:  0.4420 
Epoch [7/100] Train loss: 1.5383 Train acc: 0.4456 Val loss:  1.5099 Val acc:  0.4620 
Epoch [8/100] Train loss: 1.5111 Train acc: 0.4554 Val loss:  1.4629 Val acc:  0.4720 
Epoch [9/100] Train loss: 1.5057 Train acc: 0.4586 Val loss:  1.5973 Val acc:  0.4420 
Epoch [10/100] Train loss: 1.5013 Train acc: 0.4618 Val loss:  1.5233 Val acc:  0.4660 
Epoch [11/100] Train loss: 1.4920 Train acc: 0.4638 Val loss:  1.4447 Val acc:  0.4820 
Epoch [12/100] Train loss: 1.4916 Train a

In [12]:
perturbation_results = []

for pct in range(0, 50 + 1, 5):
    # Always start from the clean trained model
    model.load_state_dict(clean_state_dict)
    model.to(cfg.device)
    model.eval()

    flipped = flip_xnor_weights_global(
        model,
        pct=pct,
        seed=10 + pct,
    )

    test_loss, test_acc = evaluate(model, test_loader, criterion, cfg.device)

    perturbation_results.append({
        "flipped_pct": pct,
        "num_flipped": flipped,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })

    print(
        f"Flipped {pct:02d}% | "
        f"Weights flipped: {flipped} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.4f}"
    )

Flipped 00% | Weights flipped: 0 | Test Loss: 1.0373 | Test Acc: 0.6436
Flipped 05% | Weights flipped: 53709 | Test Loss: 1.2765 | Test Acc: 0.5565
Flipped 10% | Weights flipped: 107418 | Test Loss: 1.3982 | Test Acc: 0.5095
Flipped 15% | Weights flipped: 161126 | Test Loss: 1.8289 | Test Acc: 0.3444
Flipped 20% | Weights flipped: 214835 | Test Loss: 2.2580 | Test Acc: 0.3140
Flipped 25% | Weights flipped: 268544 | Test Loss: 2.2484 | Test Acc: 0.2367
Flipped 30% | Weights flipped: 322253 | Test Loss: 2.4014 | Test Acc: 0.1598
Flipped 35% | Weights flipped: 375962 | Test Loss: 2.4330 | Test Acc: 0.1350
Flipped 40% | Weights flipped: 429670 | Test Loss: 2.4708 | Test Acc: 0.1093
Flipped 45% | Weights flipped: 483379 | Test Loss: 2.4550 | Test Acc: 0.1062
Flipped 50% | Weights flipped: 537088 | Test Loss: 2.4162 | Test Acc: 0.1023
